In [ ]:
!pip install --quiet gradio

In [ ]:
import gradio as gr

from pathlib import Path

import numpy as np
import pandas as pd
import torch

from implementacion_de_la_arquitectura import DNN
from hiperparametros_optimizados import hiperparametros

from sklearn.preprocessing import StandardScaler

In [ ]:
wdir = Path(".")

N_FEATURES = 8
FEATURES = ["MedInc", "HouseAge", "AveRooms", "AveBedrms", \
            "Population", "AveOccup", \
            "Latitude", "Longitude"]
pesosfile = wdir.joinpath("estado_de_pesos.pth")

# Probamos cargar el estado del modelo entrenado y utilizarlo para predicción
modelo = DNN( input_size=N_FEATURES, \
        hidden_layers=hiperparametros["hidden_layers"], \
        dropout=hiperparametros["dropout"] )

modelo.load_state_dict(torch.load(pesosfile))
modelo.eval()

In [ ]:
data = pd.read_csv("data.csv", index_col=0)
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data)

In [ ]:
def prediccion(MedInc, HouseAge, AveRooms, AveBedrms, Population, AveOccup, \
          Latitude, Longitude):
    x = [MedInc, HouseAge, AveRooms, AveBedrms, Population, AveOccup, \
          Latitude, Longitude]
    x = pd.DataFrame([x])
    x.columns = FEATURES
    x_scaled = scaler.transform(x)
    x_scaled_tensor = torch.tensor(x_scaled, dtype=torch.float32)

    y_pred = modelo(x_scaled_tensor).detach().numpy()[0].astype(float)[0]
    # return f"El valor estimado de la vivienda en cientos de miles de dólares es: {y_pred}"
    return round(y_pred, 3)

In [ ]:
with gr.Blocks() as demo:
    MedInc = gr.Number(label="Median income in block group")
    HouseAge = gr.Number(label="Median house age in block group")
    AveRooms = gr.Number(label="Average number of rooms per household")
    AveBedrms = gr.Number(label="Average number of bedrooms per household")
    Population = gr.Number(label="Block group population")
    AveOccup = gr.Number(label="Average number of household members")
    Latitude = gr.Number(label="Block latitude")
    Longitude = gr.Number(label="Block longitude")
    MedHouseVal = gr.Number(label="El valor estimado de la vivienda es:")
    predict_btn = gr.Button("Predict")
    predict_btn.click(fn=prediccion, \
                    inputs=[MedInc, HouseAge, AveRooms, AveBedrms, \
                            Population, AveOccup, \
                            Latitude, Longitude], \
                    outputs=MedHouseVal, api_name="Predict")

In [ ]:
demo.launch(debug=True)

In [ ]:
demo = gr.Interface(
    fn=prediccion,
    inputs=["number"] * N_FEATURES,
    outputs=["text"],
    api_name="predict"
)

demo.launch(debug=True)